In [1]:
import os
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from torchvision.models import vit_b_16, ViT_B_16_Weights

from tqdm import tqdm
from sklearn.metrics import roc_auc_score

from torchmetrics.functional import structural_similarity_index_measure
import torch.nn.functional as F

In [2]:
class MVTecLOCODataset(Dataset):
    def __init__(self, root_dir, split='train', anomaly_type='logical_anomalies', transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []  # 0 for normal, 1 for anomaly

        if split == 'train':
            # Load 'good' images for training
            good_dir = os.path.join(root_dir, 'train', 'good')
            self._load_images_from_folder(good_dir, label=0)
        elif split == 'test':
            # Load 'good' and specified anomalies for testing
            good_dir = os.path.join(root_dir, 'test', 'good')
            anomaly_dir = os.path.join(root_dir, 'test', anomaly_type)
            self._load_images_from_folder(good_dir, label=0)
            self._load_images_from_folder(anomaly_dir, label=1)
        else:
            raise ValueError(f"Invalid split: {split}")

    def _load_images_from_folder(self, folder, label):
        if not os.path.isdir(folder):
            print(f"Folder {folder} is missing!")
            return
        for root, _, files in os.walk(folder):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(root, file)
                    self.image_paths.append(img_path)
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = self.labels[idx]
        return image, label, img_path


In [3]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super(PatchEmbedding, self).__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size[0] // patch_size) * (img_size[1] // patch_size)
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.position_embeddings = nn.Parameter(torch.zeros(1, self.n_patches, embed_dim))

    def forward(self, x):
        x = self.proj(x)  # Shape: (batch_size, embed_dim, n_patches_h, n_patches_w)
        x = x.flatten(2)  # Shape: (batch_size, embed_dim, n_patches)
        x = x.transpose(1, 2)  # Shape: (batch_size, n_patches, embed_dim)
        x = x + self.position_embeddings
        return x

In [4]:
class ViTEncoder(nn.Module):
    def __init__(self, embed_dim):
        super(ViTEncoder, self).__init__()
        self.vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        self.vit.heads = nn.Identity()  # Remove the classification head
        self.fc_mu = nn.Linear(self.vit.hidden_dim, embed_dim)
        self.fc_logvar = nn.Linear(self.vit.hidden_dim, embed_dim)

    def forward(self, x):
        x = self.vit(x)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar


In [5]:
class ConvDecoder(nn.Module):
    def __init__(self, img_size, in_channels, embed_dim):
        super(ConvDecoder, self).__init__()
        self.img_size = img_size  # Store the target image size
        self.embed_dim = embed_dim
        self.init_size = (img_size[0] // 16, img_size[1] // 16)
        self.fc = nn.Linear(embed_dim, 512 * self.init_size[0] * self.init_size[1])
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        batch_size = z.size(0)
        x = self.fc(z)
        x = x.view(batch_size, 512, self.init_size[0], self.init_size[1])
        x = self.decoder(x)
        # Resize to match the input image dimensions
        x = nn.functional.interpolate(x, size=self.img_size, mode='bilinear', align_corners=False)
        return x

In [6]:
class ViTVAE(nn.Module):
    def __init__(self, img_size=(224, 224), in_channels=3, embed_dim=512):
        super(ViTVAE, self).__init__()
        self.encoder = ViTEncoder(embed_dim=embed_dim)
        self.decoder = ConvDecoder(img_size, in_channels, embed_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        return x_recon, mu, logvar

In [7]:
class PerceptualLoss(nn.Module):
    def __init__(self, resize=True):
        super(PerceptualLoss, self).__init__()
        vgg = models.vgg16(pretrained=True)
        self.features = nn.Sequential(*list(vgg.features)[:16]).eval()
        for param in self.features.parameters():
            param.requires_grad = False
        self.resize = resize

    def forward(self, x, y):
        if self.resize:
            x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
            y = F.interpolate(y, size=(224, 224), mode='bilinear', align_corners=False)
        x_features = self.features(x)
        y_features = self.features(y)
        loss = F.l1_loss(x_features, y_features)
        return loss

def kl_anneal_function(epoch, start_epoch, end_epoch, beta_max):
    if epoch < start_epoch:
        return 0.0
    elif epoch > end_epoch:
        return beta_max
    else:
        return ((epoch - start_epoch) / (end_epoch - start_epoch)) * beta_max

In [8]:
def evaluate(model, dataloader, perceptual_loss_fn, beta, device):
    model.eval()
    total_loss = 0.0
    all_scores = []
    all_labels = []
    with torch.no_grad():
        for images, labels, _ in dataloader:
            images = images.to(device)
            outputs, mu, logvar = model(images)

            outputs_ssim = (outputs + 1) / 2
            images_ssim = (images + 1) / 2

            # Compute SSIM loss
            ssim_loss = 1 - structural_similarity_index_measure(outputs_ssim, images_ssim, data_range=1.0)

            # Compute Perceptual loss
            perceptual_loss = perceptual_loss_fn(outputs_ssim, images_ssim)

            # Compute KL Divergence loss
            kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)

            # Total loss
            total_sample_loss = ssim_loss + perceptual_loss + beta * kld_loss.mean()

            # Anomaly scores
            anomaly_scores = ssim_loss + perceptual_loss + beta * kld_loss

            all_scores.extend(anomaly_scores.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            total_loss += total_sample_loss.item() * images.size(0)

    avg_loss = total_loss / len(dataloader.dataset)
    auroc = roc_auc_score(all_labels, all_scores)
    return avg_loss, auroc


In [9]:
import matplotlib.pyplot as plt
def visualize_reconstruction(model, dataloader, device):
    model.eval()
    images, labels, paths = next(iter(dataloader))
    images = images.to(device)
    with torch.no_grad():
        reconstructed, _, _ = model(images)
    images = images.cpu().numpy().transpose(0, 2, 3, 1)
    reconstructed = reconstructed.cpu().numpy().transpose(0, 2, 3, 1)

    # Rescale images from [-1, 1] to [0, 1]
    images = (images + 1) / 2.0
    reconstructed = (reconstructed + 1) / 2.0

    # Ensure the pixel values are clipped between 0 and 1
    images = np.clip(images, 0, 1)
    reconstructed = np.clip(reconstructed, 0, 1)

    n = min(len(images), 4)
    plt.figure(figsize=(12, 6))
    for i in range(n):
        # Original Image
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(images[i])
        ax.axis('off')
        # Reconstructed Image
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(reconstructed[i])
        ax.axis('off')
    plt.show()

In [10]:
def train(model, dataloader, optimizer, perceptual_loss_fn, beta, device):
    model.train()
    running_loss = 0.0
    running_ssim_loss = 0.0
    running_perceptual_loss = 0.0
    running_kld_loss = 0.0

    pbar = tqdm(dataloader, desc='Training', leave=False)
    
    for images, _, _ in pbar:
        images = images.to(device)
        optimizer.zero_grad()
        outputs, mu, logvar = model(images)
        
        # Adjust images to [0, 1] range
        outputs_ssim = (outputs + 1) / 2
        images_ssim = (images + 1) / 2

        # Compute SSIM loss
        ssim_loss = 1 - structural_similarity_index_measure(outputs_ssim, images_ssim, data_range=1.0)

        # Compute Perceptual loss
        perceptual_loss = perceptual_loss_fn(outputs_ssim, images_ssim)

        # Compute KL Divergence loss
        kld_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

        # Total loss with Beta-VAE approach
        loss = ssim_loss + perceptual_loss + beta * kld_loss

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_ssim_loss += ssim_loss.item()
        running_perceptual_loss += perceptual_loss.item()
        running_kld_loss += kld_loss.item()

        pbar.set_postfix({
            'Loss': f'{loss.item():.6f}',
            'SSIM Loss': f'{ssim_loss.item():.6f}',
            'Perceptual Loss': f'{perceptual_loss.item():.6f}',
            'KL Loss': f'{kld_loss.item():.6f}',
            'Beta': f'{beta:.6f}'
        })

    avg_loss = running_loss / len(dataloader)
    avg_ssim_loss = running_ssim_loss / len(dataloader)
    avg_perceptual_loss = running_perceptual_loss / len(dataloader)
    avg_kld_loss = running_kld_loss / len(dataloader)
    return avg_loss, avg_ssim_loss, avg_perceptual_loss, avg_kld_loss


In [ ]:
def main():
    # Hyperparameters
    img_size = (224, 224)
    in_channels = 3
    embed_dim = 512
    batch_size = 16  # Adjust based on GPU memory
    num_epochs = 200  # Number of epochs
    learning_rate = 1e-4
    ROOT_DIR = "/home/hafiz/my_thesis/seg_recons/dataset/screw_bag"

    # KL Divergence parameters
    beta_max = 0.01  # Maximum beta value for Beta-VAE
    start_annealing = 0
    end_annealing = num_epochs // 2

    transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
    ])

    # Datasets and Dataloaders
    train_dataset = MVTecLOCODataset(root_dir=ROOT_DIR, split='train', transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

    test_dataset = MVTecLOCODataset(root_dir=ROOT_DIR, split='test', transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # Model, Loss, Optimizer
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ViTVAE(img_size=img_size, in_channels=in_channels, embed_dim=embed_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Perceptual Loss Function
    perceptual_loss_fn = PerceptualLoss().to(device)

    # Load the saved model and optimizer states (if any)
    checkpoint_path = 'VitVae_checkpoint.pth'
    if os.path.isfile(checkpoint_path):
        print(f"Loading checkpoint from '{checkpoint_path}'")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1  # Continue training from the next epoch
        print(f"Resuming training from epoch {start_epoch}.")
    else:
        print(f"No checkpoint found at '{checkpoint_path}'")
        start_epoch = 0  # If no checkpoint is found, start from epoch 0

    # Training Loop
    for epoch in range(start_epoch, start_epoch + num_epochs):
        beta = kl_anneal_function(epoch, start_annealing, end_annealing, beta_max)
        train_loss, train_ssim_loss, train_perceptual_loss, train_kld_loss = train(
            model, train_loader, optimizer, perceptual_loss_fn, beta, device
        )
        test_loss, auroc = evaluate(model, test_loader, perceptual_loss_fn, beta, device)

        print(
            f"Epoch {epoch+1}/{start_epoch + num_epochs}, "
            f"Train Loss: {train_loss:.6f}, SSIM Loss: {train_ssim_loss:.6f}, Perceptual Loss: {train_perceptual_loss:.6f}, KL Loss: {train_kld_loss:.6f}, "
            f"Test Loss: {test_loss:.6f}, AUROC: {auroc:.4f}"
        )

        if (epoch + 1) % 10 == 0:
            visualize_reconstruction(model, test_loader, device)

        if (epoch + 1) % 25 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, f'VitVae_checkpoint_epoch_{epoch+1}.pth')
            
    # Save the model after training
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, 'VitVae_checkpoint.pth')

if __name__ == "__main__":
    main()


In [ ]:
def main():
    # Hyperparameters
    img_size = (1024, 1024)
    patch_size = 64
    in_channels = 3
    embed_dim = 128
    depth = 6
    num_heads = 8
    batch_size = 4  # Adjust based on GPU memory
    num_epochs = 500  # Number of additional epochs you want to train
    learning_rate = 1e-4
    ROOT_DIR = "/home/hafiz/my_thesis/seg_recons/dataset/screw_bag"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    transform = transforms.Compose([
        transforms.Resize(img_size),
        #transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        #transforms.RandomRotation(15),
        #transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
    ])
    
    # Datasets and Dataloaders
    train_dataset = MVTecLOCODataset(root_dir=ROOT_DIR, split='train', transform=transform)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    test_dataset = MVTecLOCODataset(root_dir=ROOT_DIR, split='test', transform=transform)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Model, Loss, Optimizer
    
    model = ViTVAE(img_size=img_size, patch_size=patch_size, in_channels=in_channels,
                   embed_dim=embed_dim, depth=depth, num_heads=num_heads).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    # Initialize perceptual loss
    perceptual_loss = PerceptualLoss().to(device)
    #criterion = nn.MSELoss(reduction='none')  # Compute per-pixel loss

    # Load the saved model and optimizer states
    checkpoint_path = '/home/hafiz/my_thesis/seg_recons/VitVae2_checkpoint_epoch_0.pth'
    if os.path.isfile(checkpoint_path):
        print(f"Loading checkpoint from '{checkpoint_path}'")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Load model state
        model.load_state_dict(checkpoint['model_state_dict'])
        
        # Load optimizer state
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        # Set the starting epoch to the saved epoch
        start_epoch = checkpoint['epoch'] + 1  # Continue training from the next epoch
        print(f"Resuming training from epoch {start_epoch}.")
    else:
        print(f"No checkpoint found at '{checkpoint_path}'")
        start_epoch = 0  # If no checkpoint is found, start from epoch 0

    # Training Loop
    for epoch in range(start_epoch, start_epoch + num_epochs):
        kld_weight = kl_anneal_function(epoch, start_epoch, start_epoch + num_epochs // 2) * 1e-4


        train_loss, train_recon_loss, train_kld_loss = train(model, train_loader, optimizer, perceptual_loss, kld_weight, device)
        test_loss, auroc = evaluate(model, test_loader, perceptual_loss, kld_weight, device)

        print(
            f"Epoch {epoch+1}/{start_epoch + num_epochs}, "
            f"Train Loss: {train_loss:.6f}, Recon Loss: {train_recon_loss:.6f}, KL Loss: {train_kld_loss:.6f}, "
            f"Test Loss: {test_loss:.6f}, AUROC: {auroc:.4f}"
        )
        if (epoch + 1) % 10 == 0:
            visualize_reconstruction(model, test_loader, device)
        if (epoch + 1) % 25 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, f'VitVae2_ssim_checkpoint_epoch_{epoch+1}.pth')
            
    # Save the model after each epoch (optional)
    torch.save(model.state_dict(), f'VitVae2_ssim_epoch_{epoch+1}.pth')
    torch.save(optimizer.state_dict(), f'VitVae2_ssim_optim_epoch_{epoch+1}.pth')

if __name__ == "__main__":
    main()